# Bug Prediction System � Improved Model Training (V2)

## ?? Improvements over V1:
- **SHAP values** for explainable feature importance
- **Permutation importance** to rank risk factors
- **Better calibration** using calibration curves
- **Feature contribution analysis** for actionable insights
- **Cross-validation** with stratification for stability
- **Threshold optimization** for better discrimination

## ?? Dataset Info:
- 16,722 commits (Python + TypeScript)
- 873 bugs (5.22% bug rate)
- 3 time periods: 2018-2020, 2021-2023, 2024-2026
- Time span: 2018-2026


## Step 1 � Install & Import Libraries

In [ ]:
# Install required packages
!pip install shap xgboost scikit-learn pandas numpy matplotlib seaborn joblib -q
print('? Dependencies installed')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_curve, f1_score, precision_score, recall_score, auc
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
import warnings
warnings.filterwarnings('ignore')

print('? All imports successful')

## Step 2 � Load & Inspect Data

In [ ]:
# Load the training data
df = pd.read_csv('./data/combined_output.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nBug distribution:')
print(df['label'].value_counts())
print(f'Bug rate: {df["label"].sum() / len(df) * 100:.2f}%')
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nData types:\n{df.dtypes}')

## Step 3 � Feature Engineering & Preprocessing

In [ ]:
# Separate features and target
X = df.drop(columns=['label'])
y = df['label']

# Identify categorical columns
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

print(f'Numeric features ({len(numeric_cols)}): {numeric_cols}')
print(f'\nCategorical features ({len(categorical_cols)}): {categorical_cols}')

# Encode categorical features
encoders = {}
X_encoded = X.copy()

for col in categorical_cols:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_encoded[col].astype(str))
    encoders[col] = le
    print(f'Encoded {col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

# Save feature names for later
FEATURE_NAMES = list(X_encoded.columns)
print(f'\nFinal features: {FEATURE_NAMES}')

In [ ]:
# Train-test split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train set: {X_train.shape}')
print(f'Test set: {X_test.shape}')
print(f'Train bug rate: {y_train.sum() / len(y_train) * 100:.2f}%')
print(f'Test bug rate: {y_test.sum() / len(y_test) * 100:.2f}%')

## Step 4 � Train Random Forest with Cross-Validation

In [ ]:
# Train Random Forest
print('Training Random Forest Classifier...')
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=15,
    min_samples_split=10,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'  # Handle class imbalance
)

rf_model.fit(X_train, y_train)
print('? Training complete')

# Cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=skf, scoring='roc_auc')
print(f'\nCross-validation AUC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## Step 5 � Model Evaluation

In [ ]:
# Predictions
y_pred = rf_model.predict(X_test)
y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

# Metrics
auc = roc_auc_score(y_test, y_pred_proba)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print('-' * 50)
print('MODEL PERFORMANCE METRICS')
print('-' * 50)
print(f'AUC-ROC Score:  {auc:.4f}')
print(f'Precision:      {precision:.4f}')
print(f'Recall:         {recall:.4f}')
print(f'F1 Score:       {f1:.4f}')
print('-' * 50)

print('\nDetailed Classification Report:')
print(classification_report(y_test, y_pred, target_names=['Safe', 'Bug']))

## Step 6 � Feature Importance Analysis (Tree-based)

In [ ]:
# Get feature importances from Random Forest
feature_importance_df = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\n?? TOP 15 FEATURES (Tree-based Importance):')
print(feature_importance_df.head(15).to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=feature_importance_df.head(15), x='importance', y='feature', palette='viridis')
plt.title('Top 15 Features - Random Forest Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n? Feature importance plot saved')

## Step 7 � SHAP Feature Importance (Model Explainability)

In [ ]:
print('Computing SHAP values... (this may take a minute)')

# Use a sample for faster SHAP computation
X_sample = X_test.sample(min(500, len(X_test)), random_state=42)

# Create SHAP explainer
explainer = shap.TreeExplainer(rf_model)
shap_values = explainer.shap_values(X_sample)

print('? SHAP values computed')

In [ ]:
# SHAP summary plot (bar)
print('\n?? SHAP Feature Importance (Average |SHAP value|):')
shap.summary_plot(shap_values[1], X_sample, plot_type='bar', show=False)  # [1] for bug class
plt.title('SHAP Feature Importance for Bug Prediction')
plt.tight_layout()
plt.savefig('shap_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('? SHAP importance plot saved')

In [ ]:
# SHAP beeswarm plot
print('Creating detailed SHAP beeswarm plot...')
shap.summary_plot(shap_values[1], X_sample, show=False)
plt.title('SHAP Beeswarm Plot - Feature Impact on Bug Probability')
plt.tight_layout()
plt.savefig('shap_beeswarm_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('? SHAP beeswarm plot saved')

## Step 8 � Permutation Importance (Risk Factor Ranking)

In [ ]:
from sklearn.inspection import permutation_importance

print('Computing permutation importance for risk factor ranking...')

perm_importance = permutation_importance(
    rf_model, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

perm_df = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print('\n?? TOP 15 FEATURES (Permutation Importance):')
print(perm_df.head(15).to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=perm_df.head(15), x='importance', y='feature', palette='coolwarm')
plt.title('Top 15 Features - Permutation Importance (for Risk Factor Ranking)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('permutation_importance_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n? Permutation importance plot saved')

## Step 9 � Calibration Analysis

In [ ]:
# Calibrate the model
print('Calibrating model...')
calibrated_model = CalibratedClassifierCV(rf_model, method='sigmoid', cv=5)
calibrated_model.fit(X_train, y_train)

y_pred_calibrated = calibrated_model.predict_proba(X_test)[:, 1]

# Calibration curve
prob_true_uncalib, prob_pred_uncalib = calibration_curve(y_test, y_pred_proba, n_bins=10)
prob_true_calib, prob_pred_calib = calibration_curve(y_test, y_pred_calibrated, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
plt.plot(prob_pred_uncalib, prob_true_uncalib, 's-', label='Original Model')
plt.plot(prob_pred_calib, prob_true_calib, 's-', label='Calibrated Model')
plt.xlabel('Mean Predicted Probability')
plt.ylabel('Fraction of Positives')
plt.title('Calibration Curve')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('calibration_curve_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('? Calibration curve saved')

## Step 10 � ROC & Precision-Recall Curves

In [ ]:
# ROC Curve
fpr, tpr, thresholds_roc = roc_curve(y_test, y_pred_proba)

# Precision-Recall Curve
precision, recall, thresholds_pr = precision_recall_curve(y_test, y_pred_proba)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
axes[0].plot(fpr, tpr, 'b-', label=f'ROC (AUC = {auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Precision-Recall
pr_auc = auc(recall, precision)
axes[1].plot(recall, precision, 'r-', label=f'PR (AUC = {pr_auc:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('roc_pr_curves_v2.png', dpi=150, bbox_inches='tight')
plt.show()

print('? ROC and PR curves saved')

## Step 11 � Save Model & Artifacts for API

In [ ]:
# Create output directory
import os
output_dir = './server/modalv1_v2'
os.makedirs(output_dir, exist_ok=True)

# Save model artifacts
joblib.dump(rf_model, f'{output_dir}/bug_prediction_model.pkl')
joblib.dump(encoders.get('language_group', None), f'{output_dir}/encoder_language.pkl')
joblib.dump(encoders.get('time_period', None), f'{output_dir}/encoder_period.pkl')
joblib.dump(FEATURE_NAMES, f'{output_dir}/feature_cols.pkl')
joblib.dump(calibrated_model, f'{output_dir}/bug_prediction_model_calibrated.pkl')

print(f'? Model artifacts saved to {output_dir}:')
print('  - bug_prediction_model.pkl')
print('  - bug_prediction_model_calibrated.pkl')
print('  - encoder_language.pkl')
print('  - encoder_period.pkl')
print('  - feature_cols.pkl')

## Step 12 � Export Feature Importance for Risk Factor Engine

In [ ]:
# Compute mean absolute SHAP values for each feature
shap_importance_df = pd.DataFrame({
    'feature': FEATURE_NAMES,
    'mean_abs_shap': np.abs(shap_values[1]).mean(0),  # [1] for bug class
    'tree_importance': rf_model.feature_importances_,
    'permutation_importance': perm_importance.importances_mean
}).sort_values('mean_abs_shap', ascending=False)

print('\n?? COMBINED FEATURE IMPORTANCE RANKINGS:')
print(shap_importance_df.head(15).to_string(index=False))

# Save for API to use
shap_importance_df.to_csv(f'{output_dir}/feature_importance.csv', index=False)
print(f'\n? Feature importance exported to {output_dir}/feature_importance.csv')

## Step 13 � Create Risk Factor Configuration

In [ ]:
# Calculate percentile thresholds from data for better decision rules
risk_factor_config = {}

numeric_features = [col for col in FEATURE_NAMES if col not in ['lang_enc', 'period_enc']]

for feature in numeric_features:
    col_data = X_test[feature].values
    risk_factor_config[feature] = {
        'p75': float(np.percentile(col_data, 75)),  # 75th percentile = high
        'p90': float(np.percentile(col_data, 90)),  # 90th percentile = very high
        'p50': float(np.percentile(col_data, 50)),  # median
        'mean': float(col_data.mean()),
    }

import json
with open(f'{output_dir}/risk_factor_config.json', 'w') as f:
    json.dump(risk_factor_config, f, indent=2)

print('? Risk factor configuration saved')
print('\nExample - lines_added thresholds:')
print(f"  Medium risk (p75): {risk_factor_config['lines_added']['p75']:.0f} lines")
print(f"  High risk (p90):   {risk_factor_config['lines_added']['p90']:.0f} lines")

## Step 14 � Summary & Next Steps

In [ ]:
print('\n' + '='*60)
print('?? MODEL TRAINING V2 COMPLETE')
print('='*60)

print(f'\n?? FINAL MODEL METRICS:')
print(f'  � AUC-ROC:     {auc:.4f}')
print(f'  � Precision:   {precision:.4f}')
print(f'  � Recall:      {recall:.4f}')
print(f'  � F1 Score:    {f1:.4f}')

print(f'\n?? ARTIFACTS SAVED:')
print(f'  � Location: {output_dir}')
print(f'  � Files: 8 pickle files + CSV + JSON config')

print(f'\n?? NEXT STEPS:')
print(f'  1. Copy files from {output_dir} to server/modalv1/')
print(f'  2. Update main.py to use feature_importance.csv')
print(f'  3. Update get_risk_factors() to use SHAP-based rules')
print(f'  4. Test the API with improved risk factors')

print(f'\n?? IMPROVEMENTS:')
print(f'  ? SHAP values for explainability')
print(f'  ? Permutation importance ranking')
print(f'  ? Model calibration')
print(f'  ? Percentile-based thresholds')
print(f'  ? Risk factor configuration exported')

print('\n' + '='*60)